In [81]:
from utils import *
import json
ROOT.resolve()

PosixPath('/Users/lukestrange/Code/bus-tracking')

In [82]:
agencies, routes, trips, stops, stop_times, calendar, calendar_dates, shapes, feed_info = load_full_gtfs(ROOT / "war/yorkshire/190924", include=['shapes.txt', 'feed_info.txt'])

In [83]:
tt_agencies, tt_routes, tt_trips, tt_stops, tt_stop_times, tt_calendar, tt_calendar_dates = load_full_gtfs(ROOT / "18SepGB_GTFS_Timetables_Downloaded/yorkshire")

Get an example data file for the website. Use no.33 bus.

In [84]:
agency_id = 'OP931'
bus_num = '33'
date_str = "2024-09-19"

In [85]:
agency_name = agencies[agencies.agency_id == agency_id].reset_index().to_dict()['agency_name'][0]
meta = dict({'agency_name': agency_name})

In [86]:
route_id = routes[(routes.agency_id == agency_id) & (routes.route_short_name == bus_num)].route_id.values[0]

In [87]:
trips_this_route = trips[trips.route_id == route_id][['trip_id', 'trip_headsign', 'shape_id']]
unique_trips = trips_this_route.trip_id.unique()
unique_shapes = trips_this_route.shape_id.unique()
trips_this_route.count()

trip_id          57
trip_headsign    57
shape_id         57
dtype: int64

Get the timetabled trips that match the real trips

In [88]:
matched_tt_stop_times = tt_stop_times[tt_stop_times.trip_id.isin(unique_trips)]
matched_tt_stop_times_simple = matched_tt_stop_times.loc[:, ['trip_id', 'arrival_time', 'stop_id', 'stop_sequence']]
matched_tt_stop_times_simple.rename(columns={'arrival_time': 'timetable'}, inplace=True)
matched_tt_stop_times_simple

,trip_id,timetable,stop_id,stop_sequence
1935762,VJb89156b78be3588e1ea7cd4bb7d9dc3925d01144,22:30:00,450030236,0
1935763,VJb89156b78be3588e1ea7cd4bb7d9dc3925d01144,22:32:00,450010659,1
1935764,VJb89156b78be3588e1ea7cd4bb7d9dc3925d01144,22:34:00,450024000,2
1935765,VJb89156b78be3588e1ea7cd4bb7d9dc3925d01144,22:39:00,450032368,3
1935766,VJb89156b78be3588e1ea7cd4bb7d9dc3925d01144,22:41:00,450010695,4
...,...,...,...,...
1943149,VJ8088e1bf10a4d708bef69e576a4ceed4b9c3f4d9,18:16:00,450012554,50
1943150,VJ8088e1bf10a4d708bef69e576a4ceed4b9c3f4d9,18:18:00,450012506,51
1943151,VJ8088e1bf10a4d708bef69e576a4ceed4b9c3f4d9,18:20:00,450012386,52
1943152,VJ8088e1bf10a4d708bef69e576a4ceed4b9c3f4d9,18:22:00,450010668,53


In [89]:
shapes_this_route = shapes[shapes.shape_id.isin(unique_shapes)][['shape_id', 'shape_pt_lon', 'shape_pt_lat']]
shapes_this_route = shapes_this_route.groupby('shape_id').apply(lambda x: x[['shape_pt_lon', 'shape_pt_lat']].values.round(5).tolist(), include_groups=False).reset_index(name='geometry')
line = shapes_this_route.set_index('shape_id')['geometry'].to_dict()

In [90]:
stop_times_this_route = stop_times[stop_times.trip_id.isin(unique_trips)][['trip_id', 'arrival_time', 'stop_id', 'stop_sequence']]
unique_stops = stop_times_this_route.stop_id.unique()
stop_times_this_route.rename(columns={'arrival_time': 'real'}, inplace=True)

# Add the timetabled times
stop_times_this_route = stop_times_this_route.merge(matched_tt_stop_times_simple, on=['trip_id', 'stop_id', 'stop_sequence'], how='inner')

# Convert times to UTC.
stop_times_this_route['real'] = convert_to_unix_timestamp(stop_times_this_route, 'real', date_str)
stop_times_this_route['timetable'] = convert_to_unix_timestamp(stop_times_this_route, 'timetable', date_str)


stop_times_this_route
# Create an empty list to store results
trip_list = []

# Iterate over each unique trip_id
for trip_id in stop_times_this_route['trip_id'].unique():
    
    # Filter rows for the current trip_id
    trip_df = stop_times_this_route[stop_times_this_route['trip_id'] == trip_id]
    # Sort by 'real' time
    trip_df = trip_df.sort_values(by='real')
    # print(trip_df.to_csv())
    # Create a list of dicts for this trip
    current_trip_data = []
    for i, row in trip_df.iterrows():
        trip_data = {
                "stop_id": row['stop_id'],
                "real": int(row['real']),
                "timetable": int(row['timetable'])
            }
        current_trip_data.append(trip_data)
    # Append this trip's list to the main list
    trip_list.append(current_trip_data)

# Output the final list of lists
print(trip_list)

[[{'stop_id': '450010659', 'real': 1726740383, 'timetable': 1726740120}, {'stop_id': '450024000', 'real': 1726740444, 'timetable': 1726740180}, {'stop_id': '450032368', 'real': 1726740653, 'timetable': 1726740420}, {'stop_id': '450010695', 'real': 1726740832, 'timetable': 1726740540}, {'stop_id': '450011762', 'real': 1726740965, 'timetable': 1726740600}, {'stop_id': '450011770', 'real': 1726741026, 'timetable': 1726740728}, {'stop_id': '450010923', 'real': 1726741083, 'timetable': 1726740781}, {'stop_id': '450011556', 'real': 1726741128, 'timetable': 1726740848}, {'stop_id': '450010924', 'real': 1726741198, 'timetable': 1726740911}, {'stop_id': '450011565', 'real': 1726741260, 'timetable': 1726740994}, {'stop_id': '450011563', 'real': 1726741327, 'timetable': 1726741044}, {'stop_id': '450011557', 'real': 1726741449, 'timetable': 1726741233}, {'stop_id': '450011551', 'real': 1726741561, 'timetable': 1726741343}, {'stop_id': '450011571', 'real': 1726741624, 'timetable': 1726741408}, {'st

In [91]:
stops_this_route = stops[stops.stop_id.isin(unique_stops)][['stop_id', 'stop_name', 'stop_lon', 'stop_lat']]
stop_bearings = get_stop_names_and_bearings()[['stop_id', 'Bearing']]

stops_this_route = stops_this_route.merge(stop_bearings, on='stop_id', how='inner')
stops_this_route.rename(columns={'stop_name': 'name', 'stop_lat': 'lat', 'stop_lon': 'lon', 'Bearing': 'bearing'}, inplace=True)
stops_this_route['bearing'] = stops_this_route['bearing'].astype(int)
stops_this_route.set_index('stop_id', inplace=True)
stops = stops_this_route.to_dict(orient='index')

In [92]:
content = dict({'meta': meta, 'line': line, 'stops': stops, 'trips': trip_list})

In [93]:
# content = f'line:{line.strip()},stops:{stops.strip()},trips:{trip_list}'
with open(ROOT / f"web/{bus_num}.json", "w") as f:
    json.dump(content, f, separators=(',',':'))
    # json.dump(line, f, se)